# 03 - Data Validation
Deduplicate trips and apply row-level business rules. Bad rows are quarantined with a reason, not dropped.

In [ ]:
from pyspark.sql.functions import col, when

In [ ]:
trips = spark.read.table("ola_lakehouse.bronze.trips")
print(f"Bronze trip rows (including duplicates): {trips.count()}")

### Step 1 — deduplicate

In [ ]:
trips = trips.dropDuplicates(["trip_id"])
print(f"Rows after deduplication: {trips.count()}")

### Step 2 — classify rows against business rules

In [ ]:
valid_statuses = ["COMPLETED", "CANCELLED"]

trips = trips.withColumn(
    "rejection_reason",
    when(col("customer_id").isNull(), "missing_customer_id")
    .when(col("driver_id").isNull(), "missing_driver_id")
    .when(col("fare_amount") < 0, "negative_fare")
    .when(col("distance_km") < 0, "negative_distance")
    .when(~col("status").isin(valid_statuses), "invalid_status")
    .when((col("status") == "COMPLETED") & (col("drop_ts") <= col("pickup_ts")), "drop_before_pickup")
    .when(col("customer_rating").isNotNull() & ((col("customer_rating") < 1) | (col("customer_rating") > 5)), "rating_out_of_range")
    .otherwise(None)
)

trips.show()

### Step 3 — split into valid vs quarantine

In [ ]:
valid_trips = trips.filter(col("rejection_reason").isNull()).drop("rejection_reason")
quarantine_trips = trips.filter(col("rejection_reason").isNotNull())

print(f"Valid trips: {valid_trips.count()}")
print(f"Quarantined trips: {quarantine_trips.count()}")

quarantine_trips.select("trip_id","customer_id","driver_id","fare_amount","distance_km","status","rejection_reason").show()

### Step 4 — write silver outputs

In [ ]:
valid_trips.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ola_lakehouse.silver.trips")
quarantine_trips.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ola_lakehouse.silver.trips_quarantine")

In [ ]:
%sql
SELECT rejection_reason, COUNT(*) AS rejected_count
FROM ola_lakehouse.silver.trips_quarantine
GROUP BY rejection_reason
ORDER BY rejected_count DESC

### Pass dimension tables through to silver (deduplicated)

In [ ]:
for table in ["customers", "drivers", "vehicles", "locations"]:
    df = spark.read.table(f"ola_lakehouse.bronze.{table}").dropDuplicates()
    df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"ola_lakehouse.silver.{table}")

print("Dimension tables copied to silver.")